In [ ]:
# hide warnings
import warnings
warnings.filterwarnings('ignore')

# https://groups.csail.mit.edu/sls/downloads/restaurant/


In [ ]:
import pandas as pd
import json
import requests


train = pd.read_csv('train.bio', sep='\t', header=None)
train.columns = ['tokens', 'ner_tags_str']

train.head(20)

data = open('train.bio', 'r').readlines()
train_tokens = []
train_tags = []

temp_tokens = []
temp_tags = []
for line in data:
    if line != '\n':
        tag, token = line.strip().split('\t')
        temp_tokens.append(token)
        temp_tags.append(tag)
    else:
        train_tokens.append(temp_tokens)
        train_tags.append(temp_tags)
        temp_tokens = []
        temp_tags = []


In [ ]:
len(train_tokens), len(train_tags)


In [ ]:
from datasets import load_dataset, Dataset, DatasetDict

df = pd.DataFrame({'tokens': train_tokens, 'ner_tags_str': train_tags})
dataset = Dataset.from_pandas(df)
dataset = DatasetDict({'train': dataset, 'validation': dataset, 'test': dataset})
# dataset['train'][0]


In [ ]:
dataset['train'][0]


unique_tags = set()
for tags in dataset['train']['ner_tags_str']:
    unique_tags.update(tags)


unique_tags = [x[2:] for x in list(unique_tags) if x != 'O']
unique_tags = list(set(unique_tags))

tag2index = {"O": 0}
for i, tag in enumerate(unique_tags):
    tag2index[f'B-{tag}'] = len(tag2index)
    tag2index[f'I-{tag}'] = len(tag2index)

index2tag = {v: k for k, v in tag2index.items()}


In [ ]:
dataset = dataset.map(lambda example: {'ner_tags': [tag2index[tag] for tag in example['ner_tags_str']]})


In [ ]:
dataset


## Model Building

In [ ]:
from transformers import AutoTokenizer


model_ckpt = 'huawei-noah/TinyBERT_General_4L_312D'
model_ckpt = 'distilbert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)


In [ ]:
input = dataset['train'][0]['tokens']
input = tokenizer(input, is_split_into_words=True)
len(input.tokens()), len(input.word_ids())


In [ ]:
# input.tokens()


In [ ]:
def align_labels_with_tokens(labels, word_ids):
  new_labels = []
  current_word=None
  for word_id in word_ids:
    if word_id != current_word:
      current_word = word_id
      label = -100 if word_id is None else labels[word_id]
      new_labels.append(label)

    elif word_id is None:
      new_labels.append(-100)

    else:
      label = labels[word_id]

      if label%2==1:
        label = label + 1
      new_labels.append(label)

  return new_labels


In [ ]:
labels = dataset['train'][0]['ner_tags']
word_ids = input.word_ids()
# print(labels, word_ids)

# align_labels_with_tokens(labels, word_ids)




In [ ]:
def tokenize_and_align_labels(examples):
  tokenized_inputs = tokenizer(examples['tokens'], truncation=True, is_split_into_words=True)

  all_labels = examples['ner_tags']

  new_labels = []
  for i, labels in enumerate(all_labels):
    word_ids = tokenized_inputs.word_ids(i)
    new_labels.append(align_labels_with_tokens(labels, word_ids))

  tokenized_inputs['labels'] = new_labels

  return tokenized_inputs


In [ ]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)


## Data Collation and Metrics

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)


In [ ]:
# !pip install seqeval
# !pip install evaluate

import evaluate
metric = evaluate.load('seqeval')
label_names = list(tag2index)

import numpy as np

def compute_metrics(eval_preds):
  logits, labels = eval_preds

  predictions = np.argmax(logits, axis=-1)

  true_labels = [[label_names[l] for l in label if l!=-100] for label in labels]

  true_predictions = [[label_names[p] for p,l in zip(prediction, label) if l!=-100]
                      for prediction, label in zip(predictions, labels)]

  all_metrics = metric.compute(predictions=true_predictions, references=true_labels)

  return {"precision": all_metrics['overall_precision'],
          "recall": all_metrics['overall_recall'],
          "f1": all_metrics['overall_f1'],
          "accuracy": all_metrics['overall_accuracy']}


In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
                                                    model_ckpt,
                                                    id2label=index2tag,
                                                    label2id=tag2index)


In [ ]:
model.config.num_labels


In [ ]:
from transformers import TrainingArguments

args = TrainingArguments("finetuned-ner",
                         evaluation_strategy = "epoch",
                         save_strategy="epoch",
                         learning_rate = 2e-5,
                         num_train_epochs=3,
                         weight_decay=0.01)


In [ ]:
from transformers import Trainer
trainer = Trainer(model=model,
                  args=args,
                  train_dataset = tokenized_datasets['train'],
                  eval_dataset = tokenized_datasets['validation'],
                  data_collator=data_collator,
                  compute_metrics=compute_metrics,
                  tokenizer=tokenizer)

trainer.train()


In [ ]:
from transformers import pipeline

checkpoint = "finetuned-ner/checkpoint-2874"
token_classifier = pipeline(
    "token-classification", model=checkpoint, aggregation_strategy="simple"
)

token_classifier("which restaurant serves the best sushi in san francisco?")
